### Salary prediction, episode II: make it actually work (4 points)

Your main task is to use some of the tricks you've learned on the network and analyze if you can improve __validation MAE__. Try __at least 3 options__ from the list below for a passing grade. Write a short report about what you have tried. More ideas = more bonus points. 

__Please be serious:__ " plot learning curves in MAE/epoch, compare models based on optimal performance, test one change at a time. You know the drill :)

You can use either __pytorch__ or __tensorflow__ or any other framework (e.g. pure __keras__). Feel free to adapt the seminar code for your needs. For tensorflow version, consider `seminar_tf2.ipynb` as a starting point.


In [1]:
# < A whole lot of your code > - models, charts, analysis

### A short report

Please tell us what you did and how did it work.

`<YOUR_TEXT_HERE>`, i guess...

## Recommended options

#### A) CNN architecture

All the tricks you know about dense and convolutional neural networks apply here as well.
* Dropout. Nuff said.
* Batch Norm. This time it's `nn.BatchNorm*`/`L.BatchNormalization`
* Parallel convolution layers. The idea is that you apply several nn.Conv1d to the same embeddings and concatenate output channels.
* More layers, more neurons, ya know...


#### B) Play with pooling

There's more than one way to perform pooling:
* Max over time (independently for each feature)
* Average over time (excluding PAD)
* Softmax-pooling:
$$ out_{i, t} = \sum_t {h_{i,t} \cdot {{e ^ {h_{i, t}}} \over \sum_\tau e ^ {h_{j, \tau}} } }$$

* Attentive pooling
$$ out_{i, t} = \sum_t {h_{i,t} \cdot Attn(h_t)}$$

, where $$ Attn(h_t) = {{e ^ {NN_{attn}(h_t)}} \over \sum_\tau e ^ {NN_{attn}(h_\tau)}}  $$
and $NN_{attn}$ is a dense layer.

The optimal score is usually achieved by concatenating several different poolings, including several attentive pooling with different $NN_{attn}$ (aka multi-headed attention).

The catch is that keras layers do not inlude those toys. You will have to [write your own keras layer](https://keras.io/layers/writing-your-own-keras-layers/). Or use pure tensorflow, it might even be easier :)

#### C) Fun with words

It's not always a good idea to train embeddings from scratch. Here's a few tricks:

* Use a pre-trained embeddings from `gensim.downloader.load`. See last lecture.
* Start with pre-trained embeddings, then fine-tune them with gradient descent. You may or may not download pre-trained embeddings from [here](http://nlp.stanford.edu/data/glove.6B.zip) and follow this [manual](https://keras.io/examples/nlp/pretrained_word_embeddings/) to initialize your Keras embedding layer with downloaded weights.
* Use the same embedding matrix in title and desc vectorizer


#### D) Going recurrent

We've already learned that recurrent networks can do cool stuff in sequence modelling. Turns out, they're not useless for classification as well. With some tricks of course..

* Like convolutional layers, LSTM should be pooled into a fixed-size vector with some of the poolings.
* Since you know all the text in advance, use bidirectional RNN
  * Run one LSTM from left to right
  * Run another in parallel from right to left 
  * Concatenate their output sequences along unit axis (dim=-1)

* It might be good idea to mix convolutions and recurrent layers differently for title and description


#### E) Optimizing seriously

* You don't necessarily need 100 epochs. Use early stopping. If you've never done this before, take a look at [early stopping callback(keras)](https://keras.io/callbacks/#earlystopping) or in [pytorch(lightning)](https://pytorch-lightning.readthedocs.io/en/latest/common/early_stopping.html).
  * In short, train until you notice that validation
  * Maintain the best-on-validation snapshot via `model.save(file_name)`
  * Plotting learning curves is usually a good idea
  
Good luck! And may the force be with you!

In [6]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from nltk.tokenize import WordPunctTokenizer
from collections import Counter
import gc

class CNNModel(nn.Module):
    def __init__(self, n_tokens, emb_size=32, hid_size=64, dropout_prob=0.5, max_seq_length=100):
        super().__init__()
        
        self.max_seq_length = max_seq_length
        
        # Эмбеддинги для текстовых данных
        self.embedding = nn.Embedding(num_embeddings=n_tokens, embedding_dim=emb_size)
        
        # Параллельные свёрточные слои с разными размерами ядер
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=emb_size, out_channels=hid_size, kernel_size=k, padding=k//2)
            for k in [3, 5, 7]
        ])
        
        # BatchNorm слои для каждой свертки
        self.batch_norms = nn.ModuleList([
            nn.BatchNorm1d(hid_size)
            for _ in range(3)
        ])
        
        # MaxPooling для уменьшения размерности
        self.max_pool = nn.MaxPool1d(kernel_size=2)
        
        # Dropout для регуляризации
        self.dropout = nn.Dropout(p=dropout_prob)
        
        # Рассчитываем размер входа для полносвязного слоя
        # После MaxPool размер последовательности уменьшается вдвое
        pooled_length = max_seq_length // 2
        fc_input_size = hid_size * 3 * pooled_length
        
        # Полносвязные слои
        self.fc1 = nn.Linear(fc_input_size, 256)
        self.fc2 = nn.Linear(256, 1)
        
        # Инициализация весов
        self._initialize_weights()
    
    def _initialize_weights(self):
        for conv in self.convs:
            nn.init.kaiming_normal_(conv.weight)
            nn.init.constant_(conv.bias, 0)
        nn.init.xavier_normal_(self.fc1.weight)
        nn.init.constant_(self.fc1.bias, 0)
        nn.init.xavier_normal_(self.fc2.weight)
        nn.init.constant_(self.fc2.bias, 0)
    
    def forward(self, x):
        try:
            batch_size = x.size(0)
            
            # Получаем эмбеддинги и меняем размерность для Conv1d
            x = self.embedding(x)  # [batch_size, seq_len, emb_size]
            x = x.permute(0, 2, 1)  # [batch_size, emb_size, seq_len]
            
            # Применяем параллельные свертки с BatchNorm и MaxPooling
            conv_results = []
            for conv, bn in zip(self.convs, self.batch_norms):
                conv_out = conv(x)
                conv_out = bn(conv_out)
                conv_out = self.max_pool(conv_out)
                conv_results.append(conv_out)
            
            # Конкатенируем результаты всех сверток по каналам
            x = torch.cat(conv_results, dim=1)
            
            # Сглаживаем тензор для полносвязного слоя
            x = x.view(batch_size, -1)
            
            # Применяем полносвязные слои
            x = self.dropout(x)
            x = torch.relu(self.fc1(x))
            x = self.dropout(x)
            x = self.fc2(x)
            
            return x.squeeze(1)
            
        except RuntimeError as e:
            print(f"Ошибка в forward pass: {str(e)}")
            print(f"Размерности входного тензора: {x.size()}")
            raise e

class TextDataset(Dataset):
    def __init__(self, data, token_to_index, max_seq_length):
        self.data = data
        self.token_to_index = token_to_index
        self.max_seq_length = max_seq_length
        self.tokenizer = WordPunctTokenizer()
        
    def __len__(self):
        return len(self.data)
    
    def preprocess_text(self, text):
        text = str(text)
        return " ".join(self.tokenizer.tokenize(text)).lower()
    
    def text_to_indices(self, text):
        text = self.preprocess_text(text)
        return [self.token_to_index.get(word, self.token_to_index['UNK']) 
                for word in text.split()]
    
    def pad_sequence(self, sequence):
        if len(sequence) > self.max_seq_length:
            return sequence[:self.max_seq_length]
        return sequence + [self.token_to_index['PAD']] * (self.max_seq_length - len(sequence))
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        title_indices = self.pad_sequence(self.text_to_indices(row['Title']))
        desc_indices = self.pad_sequence(self.text_to_indices(row['FullDescription']))
        
        return {
            'title': torch.tensor(title_indices, dtype=torch.long),
            'description': torch.tensor(desc_indices, dtype=torch.long),
            'target': torch.tensor(row['Log1pSalary'], dtype=torch.float32)
        }

def create_vocabulary(data, min_count=10):
    token_counts = Counter()
    tokenizer = WordPunctTokenizer()
    
    # Обрабатываем данные батчами
    batch_size = 1000
    for i in range(0, len(data), batch_size):
        batch = data.iloc[i:i+batch_size]
        for column in ['Title', 'FullDescription']:
            for text in batch[column]:
                text = str(text)
                tokens = tokenizer.tokenize(text.lower())
                token_counts.update(tokens)
    
    tokens = [token for token, count in token_counts.items() if count >= min_count]
    tokens = ['UNK', 'PAD'] + tokens
    return {token: idx for idx, token in enumerate(tokens)}

def train_model(model, train_loader, criterion, optimizer, device, num_epochs=10):
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        total_mae = 0  # Добавляем total_mae = 0 в начало эпохи
        for batch in train_loader:
            title = batch['title'].to(device)
            desc = batch['description'].to(device)
            target = batch['target'].to(device)
            
            optimizer.zero_grad()
            
            # Объединяем title и description
            X_batch = torch.cat([title, desc], dim=1)
            
            output = model(X_batch)
            loss = criterion(output, target)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            # Переводим предсказания и истинные значения обратно в исходную размерность
            original_pred = torch.expm1(output)
            original_target = torch.expm1(target)

            # Считаем MAE в исходной размерности
            total_mae = torch.mean(torch.abs(original_pred - original_target)).item()
            
            # Считаем MAE для текущего батча
            mae = torch.mean(torch.abs(output - target)).item()
            total_mae += mae 

        # В конце эпохи, где печатается loss, добавьте:
        print(f"Epoch {epoch+1}, Loss: {(total_loss/len(train_loader)):.2}, MAE: {(total_mae/len(train_loader)):.2}")
        
        # Очищаем кэш CUDA после каждой эпохи
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        # Принудительная сборка мусора
        gc.collect()

# Основной код
def main():
    # Загружаем данные частями
    chunk_size = 10000
    data_iterator = pd.read_csv("data/Train_rev1.zip", 
                               compression='zip', 
                               chunksize=chunk_size)
    
    # Берем первый чанк для создания словаря
    first_chunk = next(data_iterator)
    
    # Добавляем Log1pSalary
    first_chunk['Log1pSalary'] = np.log1p(first_chunk['SalaryNormalized']).astype('float32')
    
    token_to_index = create_vocabulary(first_chunk)
    
    # Создаем датасет и загружаем данные батчами
    dataset = TextDataset(first_chunk, token_to_index, max_seq_length=100)
    train_loader = DataLoader(dataset, batch_size=128, shuffle=True)
    
    # Инициализация модели
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    
    max_seq_length = 100  # Должно совпадать с max_seq_length в TextDataset
    
    # Инициализация модели с явным указанием max_seq_length
    model = CNNModel(
        n_tokens=len(token_to_index),
        emb_size=32,
        hid_size=64,
        dropout_prob=0.5,
        max_seq_length=max_seq_length * 2  # Умножаем на 2, так как конкатенируем title и description
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Обучение модели
    train_model(model, train_loader, criterion, optimizer, device)


if __name__ == "__main__":
    main()

Epoch 1, Loss: 210.84513275532782, MAE: 380.9469838927064
Epoch 2, Loss: 103.8811312325393, MAE: 307.2236266317247
Epoch 3, Loss: 102.5986820655533, MAE: 379.38882484918906
Epoch 4, Loss: 101.21947981436041, MAE: 390.5470857258085
Epoch 5, Loss: 99.82567074932629, MAE: 445.59698107272766
Epoch 6, Loss: 98.36987662013573, MAE: 396.0475505152835
Epoch 7, Loss: 96.94857247268098, MAE: 407.26626194579694
Epoch 8, Loss: 95.4915232598027, MAE: 392.85348938084854
Epoch 9, Loss: 94.0607923676696, MAE: 384.95314692243744
Epoch 10, Loss: 92.65634367737589, MAE: 415.9169303556032


In [9]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from nltk.tokenize import WordPunctTokenizer
from collections import Counter
import gc

class RNNModel(nn.Module):
    def __init__(self, n_tokens, emb_size=32, hid_size=64, dropout_prob=0.5, max_seq_length=100):
        super().__init__()
        
        self.max_seq_length = max_seq_length
        
        # Эмбеддинги для текстовых данных
        self.embedding = nn.Embedding(num_embeddings=n_tokens, embedding_dim=emb_size)
        
        # LSTM слой
        self.rnn = nn.LSTM(
            input_size=emb_size,
            hidden_size=hid_size,
            num_layers=1,
            batch_first=True,
            bidirectional=False
        )
        
        # Dropout для регуляризации
        self.dropout = nn.Dropout(p=dropout_prob)
        
        # Полносвязные слои
        self.fc1 = nn.Linear(hid_size, 256)
        self.fc2 = nn.Linear(256, 1)
        
        # Инициализация весов
        self._initialize_weights()
    
    def _initialize_weights(self):
        for name, param in self.rnn.named_parameters():
            if 'weight' in name:
                nn.init.kaiming_normal_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0)
        nn.init.xavier_normal_(self.fc1.weight)
        nn.init.constant_(self.fc1.bias, 0)
        nn.init.xavier_normal_(self.fc2.weight)
        nn.init.constant_(self.fc2.bias, 0)
    
    def forward(self, x):
        try:
            batch_size = x.size(0)
            
            # Получаем эмбеддинги
            x = self.embedding(x)  # [batch_size, seq_len, emb_size]
            
            # Применяем LSTM
            rnn_out, _ = self.rnn(x)  # rnn_out: [batch_size, seq_len, hid_size]
            
            # Берем последний скрытый состояние
            last_hidden = rnn_out[:, -1, :]  # [batch_size, hid_size]
            
            # Применяем полносвязные слои
            x = self.dropout(last_hidden)
            x = torch.relu(self.fc1(x))
            x = self.dropout(x)
            x = self.fc2(x)
            
            return x.squeeze(1)
            
        except RuntimeError as e:
            print(f"Ошибка в forward pass: {str(e)}")
            print(f"Размерности входного тензора: {x.size()}")
            raise e

class TextDataset(Dataset):
    def __init__(self, data, token_to_index, max_seq_length):
        self.data = data
        self.token_to_index = token_to_index
        self.max_seq_length = max_seq_length
        self.tokenizer = WordPunctTokenizer()
        
    def __len__(self):
        return len(self.data)
    
    def preprocess_text(self, text):
        text = str(text)
        return " ".join(self.tokenizer.tokenize(text)).lower()
    
    def text_to_indices(self, text):
        text = self.preprocess_text(text)
        return [self.token_to_index.get(word, self.token_to_index['UNK']) 
                for word in text.split()]
    
    def pad_sequence(self, sequence):
        if len(sequence) > self.max_seq_length:
            return sequence[:self.max_seq_length]
        return sequence + [self.token_to_index['PAD']] * (self.max_seq_length - len(sequence))
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        title_indices = self.pad_sequence(self.text_to_indices(row['Title']))
        desc_indices = self.pad_sequence(self.text_to_indices(row['FullDescription']))
        
        return {
            'title': torch.tensor(title_indices, dtype=torch.long),
            'description': torch.tensor(desc_indices, dtype=torch.long),
            'target': torch.tensor(row['Log1pSalary'], dtype=torch.float32)
        }

def create_vocabulary(data, min_count=10):
    token_counts = Counter()
    tokenizer = WordPunctTokenizer()
    
    # Обрабатываем данные батчами
    batch_size = 1000
    for i in range(0, len(data), batch_size):
        batch = data.iloc[i:i+batch_size]
        for column in ['Title', 'FullDescription']:
            for text in batch[column]:
                text = str(text)
                tokens = tokenizer.tokenize(text.lower())
                token_counts.update(tokens)
    
    tokens = [token for token, count in token_counts.items() if count >= min_count]
    tokens = ['UNK', 'PAD'] + tokens
    return {token: idx for idx, token in enumerate(tokens)}

def train_model(model, train_loader, criterion, optimizer, device, num_epochs=10):
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        total_mae = 0  # Добавляем total_mse = 0 в начало эпохи
        for batch in train_loader:
            title = batch['title'].to(device)
            desc = batch['description'].to(device)
            target = batch['target'].to(device)
            
            optimizer.zero_grad()
            
            # Объединяем title и description
            X_batch = torch.cat([title, desc], dim=1)
            
            output = model(X_batch)
            loss = criterion(output, target)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            # Переводим предсказания и истинные значения обратно в исходную размерность
            original_pred = torch.expm1(output)
            original_target = torch.expm1(target)
            # Считаем MSE для текущего батча
            mae = torch.mean((original_pred - original_target)).item()
            total_mae += mae 

        # В конце эпохи, где печатается loss, добавьте:
        print(f"Epoch {epoch+1}, Loss: {(total_loss/len(train_loader)):.2}, MAE: {(total_mae/len(train_loader)):.2}")
        
        # Очищаем кэш CUDA после каждой эпохи
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        # Принудительная сборка мусора
        gc.collect()

# Основной код
def main():
    # Загружаем данные частями
    chunk_size = 10000
    data_iterator = pd.read_csv("data/Train_rev1.zip", 
                               compression='zip', 
                               chunksize=chunk_size)
    
    # Берем первый чанк для создания словаря
    first_chunk = next(data_iterator)
    
    # Добавляем Log1pSalary
    first_chunk['Log1pSalary'] = np.log1p(first_chunk['SalaryNormalized']).astype('float32')
    
    token_to_index = create_vocabulary(first_chunk)
    
    # Создаем датасет и загружаем данные батчами
    dataset = TextDataset(first_chunk, token_to_index, max_seq_length=100)
    train_loader = DataLoader(dataset, batch_size=128, shuffle=True)
    
    # Инициализация модели
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    max_seq_length = 100  # Должно совпадать с max_seq_length в TextDataset
    
    # Инициализация модели с явным указанием max_seq_length
    model = RNNModel(
        n_tokens=len(token_to_index),
        emb_size=32,
        hid_size=64,
        dropout_prob=0.5,
        max_seq_length=max_seq_length * 2  # Умножаем на 2, так как конкатенируем title и description
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Обучение модели
    train_model(model, train_loader, criterion, optimizer, device)


if __name__ == "__main__":
    main()

Epoch 1, Loss: 4.2e+01, MSE: 2.5e+06
Epoch 2, Loss: 4.3, MSE: 1.8e+05
Epoch 3, Loss: 3.4, MSE: 1.1e+05
Epoch 4, Loss: 2.8, MSE: 5.7e+04
Epoch 5, Loss: 2.4, MSE: 4.4e+04
Epoch 6, Loss: 2.1, MSE: 3.2e+04
Epoch 7, Loss: 2.0, MSE: 2.7e+04
Epoch 8, Loss: 1.9, MSE: 2.5e+04
Epoch 9, Loss: 1.7, MSE: 2e+04
Epoch 10, Loss: 1.7, MSE: 2e+04
